In [ ]:

from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import spearmanr, norm
from statsmodels.stats.multitest import multipletests
from IPython.display import display

warnings.filterwarnings("ignore")

BASE_DIR = Path("data")
CTRP_DIR = BASE_DIR / "CTRP"

SCREEN_DIR = BASE_DIR / "celligner_mss_tmbh_pan_cancer_screen"
OUTPUT_DIR = SCREEN_DIR / "CTRP_validation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CURVES_FILE = CTRP_DIR / "v20.data.curves_post_qc.txt"
EXPERIMENT_FILE = CTRP_DIR / "v20.meta.per_experiment.txt"
CELL_LINE_FILE = CTRP_DIR / "v20.meta.per_cell_line.txt"
COMPOUND_FILE = CTRP_DIR / "v20.meta.per_compound.txt"
RECALCULATED_AUC_FILE = CTRP_DIR / "reCalculatedAUCs.txt"

CRC_PRED_FILE = SCREEN_DIR / "crc_all_cell_line_predictions.csv"
UCEC_PRED_FILE = SCREEN_DIR / "ucec_all_cell_line_predictions.csv"
COMBINED_PRED_FILE = SCREEN_DIR / "cell_line_subgroup_predictions.csv"

GDSC_RESULT_CANDIDATES = [
    SCREEN_DIR / "GDSC_validation" / "gdsc_continuous_tissue_adjusted_results.csv",
    SCREEN_DIR / "GDSC_validation" / "gdsc_significant_results.csv",
]

PRISM_RESULT_CANDIDATES = [
    SCREEN_DIR / "pan_cancer_tissue_adjusted_continuous_results.csv",
    SCREEN_DIR / "all_drug_continuous_probability_results.csv",
]

SCORE_COLUMN = "p_MSS_TMBH"
MIN_MODELS_PER_DRUG = 20
MIN_TISSUES_PER_DRUG = 3

print("CTRP directory:", CTRP_DIR)
print("Output directory:", OUTPUT_DIR)


In [ ]:

def read_table_auto(path):
    if not path.exists():
        raise FileNotFoundError(path)

    return pd.read_csv(
        path,
        sep=None,
        engine="python",
        
    )


def normalize_colname(x):
    return re.sub(r"[^a-z0-9]+", "", str(x).lower())


def find_col(df, candidates, required=True):
    exact = {str(c).lower(): c for c in df.columns}

    for candidate in candidates:
        if candidate.lower() in exact:
            return exact[candidate.lower()]

    normalized = {
        normalize_colname(c): c
        for c in df.columns
    }

    for candidate in candidates:
        key = normalize_colname(candidate)
        if key in normalized:
            return normalized[key]

    if required:
        raise KeyError(
            f"Could not find any of {candidates}. "
            f"Available columns: {df.columns.tolist()}"
        )

    return None


def normalize_cell_name(x):
    if pd.isna(x):
        return pd.NA

    return re.sub(
        r"[^A-Z0-9]+",
        "",
        str(x).upper().strip(),
    )


def normalize_drug_name(x):
    if pd.isna(x):
        return pd.NA

    x = str(x).lower().strip()

    aliases = {
        "pd-0325901": "pd0325901",
        "pd 0325901": "pd0325901",
        "azd-6244": "selumetinib",
    }

    x = aliases.get(x, x)

    return re.sub(r"[^a-z0-9]+", "", x)


def first_existing(paths):
    for path in paths:
        if path.exists():
            return path
    return None


In [ ]:

def load_predictions():
    parts = []

    if CRC_PRED_FILE.exists():
        x = pd.read_csv(CRC_PRED_FILE)
        if "classifier_source" not in x.columns:
            x["classifier_source"] = "CRC"
        parts.append(x)

    if UCEC_PRED_FILE.exists():
        x = pd.read_csv(UCEC_PRED_FILE)
        if "classifier_source" not in x.columns:
            x["classifier_source"] = "UCEC"
        parts.append(x)

    if parts:
        pred = pd.concat(parts, ignore_index=True)

    elif COMBINED_PRED_FILE.exists():
        pred = pd.read_csv(COMBINED_PRED_FILE)

    else:
        raise FileNotFoundError(
            f"No prediction files found in {SCREEN_DIR}"
        )

    if "DepMap_ID" not in pred.columns:
        for c in ["ModelID", "depmap_id", "DepMapID"]:
            if c in pred.columns:
                pred = pred.rename(columns={c: "DepMap_ID"})
                break

    required = {"DepMap_ID", "classifier_source", SCORE_COLUMN}
    missing = required - set(pred.columns)

    if missing:
        raise KeyError(
            f"Prediction table missing columns: {sorted(missing)}"
        )

    pred["classifier_source"] = (
        pred["classifier_source"]
        .astype(str)
        .str.upper()
        .replace({"COADREAD": "CRC"})
    )

    return (
        pred.sort_values(["classifier_source", "DepMap_ID"])
        .drop_duplicates(["classifier_source", "DepMap_ID"])
        .reset_index(drop=True)
    )


predictions = load_predictions()

TISSUE_CANDIDATES = [
    "OncotreeLineage",
    "OncotreePrimaryDisease",
    "PrimaryDisease",
    "lineage",
    "Lineage",
    "OncotreeCode",
]

TISSUE_COLUMN = next(
    (c for c in TISSUE_CANDIDATES if c in predictions.columns),
    None,
)

display(
    predictions.groupby("classifier_source").agg(
        n_models=("DepMap_ID", "nunique"),
        score_min=(SCORE_COLUMN, "min"),
        score_median=(SCORE_COLUMN, "median"),
        score_max=(SCORE_COLUMN, "max"),
    )
)

print("Detected tissue column:", TISSUE_COLUMN)


In [ ]:
curves_raw = read_table_auto(CURVES_FILE)
experiments_raw = read_table_auto(EXPERIMENT_FILE)
cell_lines_raw = read_table_auto(CELL_LINE_FILE)
compounds_raw = read_table_auto(COMPOUND_FILE)

print("Curves:", curves_raw.shape)
print(curves_raw.columns.tolist())

print("\nExperiments:", experiments_raw.shape)
print(experiments_raw.columns.tolist())

print("\nCell lines:", cell_lines_raw.shape)
print(cell_lines_raw.columns.tolist())

print("\nCompounds:", compounds_raw.shape)
print(compounds_raw.columns.tolist())

display(curves_raw.head())
display(experiments_raw.head())
display(cell_lines_raw.head())
display(compounds_raw.head())


In [ ]:

if RECALCULATED_AUC_FILE.exists():
    recalculated_auc = read_table_auto(RECALCULATED_AUC_FILE)
    print("reCalculatedAUCs:", recalculated_auc.shape)
    print(recalculated_auc.columns.tolist())
    display(recalculated_auc.head())
else:
    recalculated_auc = None


In [ ]:

curve_experiment_col = find_col(
    curves_raw,
    ["experiment_id", "experimentid"],
)

curve_compound_col = find_col(
    curves_raw,
    ["master_cpd_id", "mastercpdid", "compound_id", "cpd_id"],
)

curve_auc_col = find_col(
    curves_raw,
    [
        "area_under_curve",
        "area under curve",
        "auc",
        "curve_auc",
    ],
)

print("Using curve columns:")
print(" experiment:", curve_experiment_col)
print(" compound:", curve_compound_col)
print(" AUC:", curve_auc_col)

curves = pd.DataFrame({
    "experiment_id": curves_raw[curve_experiment_col],
    "master_cpd_id": curves_raw[curve_compound_col],
    "auc": pd.to_numeric(
        curves_raw[curve_auc_col],
        errors="coerce",
    ),
}).dropna(
    subset=["experiment_id", "master_cpd_id", "auc"]
)

curves = (
    curves.groupby(
        ["experiment_id", "master_cpd_id"],
        as_index=False,
        dropna=False,
    )
    .agg(auc=("auc", "median"))
)

display(curves.head())


In [ ]:

exp_id_col = find_col(
    experiments_raw,
    ["experiment_id", "experimentid"],
)

exp_ccl_col = find_col(
    experiments_raw,
    ["master_ccl_id", "mastercclid", "ccl_id", "cell_line_id"],
)

experiments = pd.DataFrame({
    "experiment_id": experiments_raw[exp_id_col],
    "master_ccl_id": experiments_raw[exp_ccl_col],
}).drop_duplicates()

ambiguity = (
    experiments.groupby("experiment_id")["master_ccl_id"]
    .nunique()
)

if (ambiguity > 1).any():
    raise ValueError(
        "Some CTRP experiment IDs map to multiple cell lines."
    )

experiments = experiments.drop_duplicates("experiment_id")
display(experiments.head())


In [ ]:

ccl_id_col = find_col(
    cell_lines_raw,
    ["master_ccl_id", "mastercclid", "ccl_id"],
)

ccl_name_col = find_col(
    cell_lines_raw,
    [
        "ccl_name",
        "ccle_name",
        "cell_line_name",
        "cell_line",
        "name",
    ],
)

ctrp_tissue_col = find_col(
    cell_lines_raw,
    [
        "ccle_primary_site",
        "primary_site",
        "lineage",
        "tissue",
        "site_primary",
    ],
    required=False,
)

cell_lines = pd.DataFrame({
    "master_ccl_id": cell_lines_raw[ccl_id_col],
    "ctrp_cell_line": cell_lines_raw[ccl_name_col].astype(str),
})

if ctrp_tissue_col is not None:
    cell_lines["ctrp_tissue"] = cell_lines_raw[ctrp_tissue_col]
else:
    cell_lines["ctrp_tissue"] = pd.NA

cell_lines["ctrp_name_norm"] = (
    cell_lines["ctrp_cell_line"].map(normalize_cell_name)
)

cell_lines = cell_lines.drop_duplicates("master_ccl_id")

print("CTRP tissue metadata column:", ctrp_tissue_col)
display(cell_lines.head())


In [ ]:

cpd_id_col = find_col(
    compounds_raw,
    ["master_cpd_id", "mastercpdid", "cpd_id", "compound_id"],
)

cpd_name_col = find_col(
    compounds_raw,
    [
        "cpd_name",
        "compound_name",
        "drug_name",
        "name",
    ],
)

target_col = find_col(
    compounds_raw,
    [
        "gene_symbol_of_protein_target",
        "gene_symbol",
        "target",
        "targets",
        "protein_target",
    ],
    required=False,
)

broad_id_col = find_col(
    compounds_raw,
    ["broad_cpd_id", "broad_id"],
    required=False,
)

compounds = pd.DataFrame({
    "master_cpd_id": compounds_raw[cpd_id_col],
    "drug": compounds_raw[cpd_name_col].astype(str),
})

compounds["target"] = (
    compounds_raw[target_col]
    if target_col is not None
    else pd.NA
)

compounds["broad_cpd_id"] = (
    compounds_raw[broad_id_col]
    if broad_id_col is not None
    else pd.NA
)

compounds["drug_key"] = (
    compounds["drug"].map(normalize_drug_name)
)

compounds = compounds.drop_duplicates("master_cpd_id")

display(compounds.head())


In [ ]:

ctrp = (
    curves
    .merge(
        experiments,
        on="experiment_id",
        how="inner",
        validate="many_to_one",
    )
    .merge(
        cell_lines,
        on="master_ccl_id",
        how="inner",
        validate="many_to_one",
    )
    .merge(
        compounds,
        on="master_cpd_id",
        how="left",
        validate="many_to_one",
    )
)

ctrp = (
    ctrp.groupby(
        [
            "master_ccl_id",
            "ctrp_cell_line",
            "ctrp_name_norm",
            "ctrp_tissue",
            "master_cpd_id",
            "drug",
            "drug_key",
            "target",
            "broad_cpd_id",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        auc=("auc", "median"),
        n_experiments=("experiment_id", "nunique"),
    )
)

print("CTRP model-drug table:", ctrp.shape)

display(
    pd.DataFrame({
        "n_cell_lines": [ctrp["master_ccl_id"].nunique()],
        "n_compounds": [ctrp["master_cpd_id"].nunique()],
        "n_response_rows": [len(ctrp)],
        "auc_min": [ctrp["auc"].min()],
        "auc_median": [ctrp["auc"].median()],
        "auc_max": [ctrp["auc"].max()],
    })
)


In [ ]:

DEPMAP_MODEL_FILE = None





def score_model_metadata_file(path):
    try:
        sample = pd.read_csv(path, nrows=5)
    except Exception:
        return -1

    cols = set(sample.columns)
    score = 0

    if any(c in cols for c in ["ModelID", "DepMap_ID", "DepMapID"]):
        score += 10

    if any(
        c in cols
        for c in [
            "StrippedCellLineName",
            "CellLineName",
            "CCLEName",
            "ModelName",
        ]
    ):
        score += 5

    if any(
        c in cols
        for c in [
            "OncotreeLineage",
            "OncotreePrimaryDisease",
            "PrimaryDisease",
        ]
    ):
        score += 2

    return score


def auto_find_depmap_model_file():
    candidates = []

    roots = [
        BASE_DIR,
        Path("combined_paper"),
    ]

    patterns = [
        "**/Model.csv",
        "**/model.csv",
        "**/*Model*.csv",
        "**/*model_metadata*.csv",
    ]

    seen = set()

    for root in roots:
        if not root.exists():
            continue

        for pattern in patterns:
            for path in root.glob(pattern):
                if path in seen:
                    continue

                seen.add(path)

                if "CTRP_validation" in str(path):
                    continue

                score = score_model_metadata_file(path)

                if score > 0:
                    candidates.append((score, path))

    if not candidates:
        return None

    candidates.sort(
        key=lambda x: (-x[0], len(str(x[1])))
    )

    print("Candidate model metadata files:")

    for score, path in candidates[:10]:
        print(f" score={score:2d}  {path}")

    return candidates[0][1]


if DEPMAP_MODEL_FILE is None:
    DEPMAP_MODEL_FILE = auto_find_depmap_model_file()

if DEPMAP_MODEL_FILE is None:
    raise FileNotFoundError(
        "Could not find DepMap model metadata. "
        "Set DEPMAP_MODEL_FILE manually."
    )

print("\nUsing:", DEPMAP_MODEL_FILE)

models = pd.read_csv(DEPMAP_MODEL_FILE)
print(models.shape)
print(models.columns.tolist())
display(models.head())


In [ ]:

def build_name_crosswalk(models):
    depmap_col = find_col(
        models,
        ["ModelID", "DepMap_ID", "DepMapID"],
    )

    name_cols = [
        c for c in [
            "StrippedCellLineName",
            "CellLineName",
            "CCLEName",
            "ModelName",
        ]
        if c in models.columns
    ]

    if not name_cols:
        raise KeyError(
            "No recognizable cell-line name columns in DepMap metadata."
        )

    tissue_col = next(
        (
            c for c in [
                "OncotreeLineage",
                "OncotreePrimaryDisease",
                "PrimaryDisease",
                "lineage",
                "Lineage",
                "OncotreeCode",
            ]
            if c in models.columns
        ),
        None,
    )

    records = []

    for _, row in models.iterrows():
        depmap_id = row[depmap_col]

        if pd.isna(depmap_id):
            continue

        tissue = (
            row[tissue_col]
            if tissue_col is not None
            else pd.NA
        )

        names = []

        for c in name_cols:
            value = row[c]

            if pd.isna(value):
                continue

            names.append(normalize_cell_name(value))

            if c == "CCLEName" and "_" in str(value):
                names.append(
                    normalize_cell_name(
                        str(value).split("_", 1)[0]
                    )
                )

        names = list(
            dict.fromkeys(
                x for x in names
                if pd.notna(x)
            )
        )

        for name in names:
            records.append({
                "DepMap_ID": str(depmap_id),
                "depmap_name_norm": name,
                "model_tissue": tissue,
            })

    return pd.DataFrame(records).drop_duplicates()


depmap_names = build_name_crosswalk(models)

name_counts = (
    depmap_names.groupby("depmap_name_norm")["DepMap_ID"]
    .nunique()
)

valid_names = name_counts[
    name_counts == 1
].index

name_lookup = (
    depmap_names[
        depmap_names["depmap_name_norm"].isin(valid_names)
    ]
    .drop_duplicates("depmap_name_norm")
    .set_index("depmap_name_norm")["DepMap_ID"]
    .to_dict()
)

ctrp["DepMap_ID"] = (
    ctrp["ctrp_name_norm"].map(name_lookup)
)

mapping_summary = (
    ctrp[
        [
            "master_ccl_id",
            "ctrp_cell_line",
            "DepMap_ID",
        ]
    ]
    .drop_duplicates()
)

n_total = mapping_summary["master_ccl_id"].nunique()
n_mapped = mapping_summary["DepMap_ID"].nunique()

print("CTRP cell lines:", n_total)
print("Mapped to DepMap:", n_mapped)
print("Mapping fraction:", n_mapped / n_total)

unmapped = (
    mapping_summary[
        mapping_summary["DepMap_ID"].isna()
    ]
    .sort_values("ctrp_cell_line")
)

display(unmapped.head(100))

unmapped.to_csv(
    OUTPUT_DIR / "unmapped_ctrp_cell_lines.csv",
    index=False,
)

ctrp_mapped = ctrp.dropna(subset=["DepMap_ID"]).copy()


In [ ]:

model_tissue = (
    depmap_names[
        ["DepMap_ID", "model_tissue"]
    ]
    .dropna()
    .drop_duplicates()
)

tissue_counts = (
    model_tissue.groupby("DepMap_ID")["model_tissue"]
    .nunique()
)

valid = tissue_counts[
    tissue_counts == 1
].index

model_tissue = (
    model_tissue[
        model_tissue["DepMap_ID"].isin(valid)
    ]
    .drop_duplicates("DepMap_ID")
)

if TISSUE_COLUMN is None:
    predictions = predictions.merge(
        model_tissue,
        on="DepMap_ID",
        how="left",
        validate="many_to_one",
    )

    TISSUE_COLUMN = "model_tissue"

print("Using tissue column:", TISSUE_COLUMN)

display(
    predictions.groupby("classifier_source").agg(
        n_models=("DepMap_ID", "nunique"),
        n_tissues=(TISSUE_COLUMN, "nunique"),
    )
)


In [ ]:

score_cols = [
    "DepMap_ID",
    "classifier_source",
    SCORE_COLUMN,
    TISSUE_COLUMN,
]

for optional in [
    "p_MSS_TMBL",
    "p_MSIH",
    "probability_margin",
    "predicted_group",
]:
    if optional in predictions.columns:
        score_cols.append(optional)

score_table = (
    predictions[score_cols]
    .drop_duplicates(
        ["classifier_source", "DepMap_ID"]
    )
)

parts = []

for classifier_source, score_df in score_table.groupby(
    "classifier_source",
    sort=False,
):
    x = ctrp_mapped.merge(
        score_df,
        on="DepMap_ID",
        how="inner",
        validate="many_to_one",
    )

    parts.append(x)

ctrp_scores = pd.concat(
    parts,
    ignore_index=True,
)

display(
    ctrp_scores.groupby("classifier_source").agg(
        n_models=("DepMap_ID", "nunique"),
        n_compounds=("master_cpd_id", "nunique"),
        n_tissues=(TISSUE_COLUMN, "nunique"),
        n_measurements=("auc", "size"),
    )
)

ctrp_scores.to_csv(
    OUTPUT_DIR / "ctrp_model_score_merged.csv",
    index=False,
)


In [ ]:

def run_ctrp_continuous_screen(
    merged,
    score=SCORE_COLUMN,
    min_models=MIN_MODELS_PER_DRUG,
    min_tissues=MIN_TISSUES_PER_DRUG,
):
    rows = []

    for (
        classifier_source,
        master_cpd_id,
        drug,
    ), drug_df in merged.groupby(
        [
            "classifier_source",
            "master_cpd_id",
            "drug",
        ],
        sort=False,
    ):
        data = drug_df[
            [
                "DepMap_ID",
                TISSUE_COLUMN,
                score,
                "auc",
            ]
        ].dropna().copy()

        data = data.drop_duplicates("DepMap_ID")

        tissue_n = (
            data.groupby(TISSUE_COLUMN)["DepMap_ID"]
            .nunique()
        )

        eligible_tissues = tissue_n[
            tissue_n >= 2
        ].index

        data = data[
            data[TISSUE_COLUMN].isin(eligible_tissues)
        ].copy()

        n_models = data["DepMap_ID"].nunique()
        n_tissues = data[TISSUE_COLUMN].nunique()

        if n_models < min_models:
            continue

        if n_tissues < min_tissues:
            continue

        data["score_within_tissue"] = (
            data[score]
            - data.groupby(TISSUE_COLUMN)[score]
            .transform("mean")
        )

        data["auc_within_tissue"] = (
            data["auc"]
            - data.groupby(TISSUE_COLUMN)["auc"]
            .transform("mean")
        )

        if np.isclose(
            data["score_within_tissue"].abs().sum(),
            0,
        ):
            continue

        fit = sm.OLS(
            data["auc_within_tissue"].astype(float),
            data[["score_within_tissue"]].astype(float),
        ).fit(cov_type="HC3")

        rho, rho_p = spearmanr(
            data["score_within_tissue"],
            data["auc_within_tissue"],
            nan_policy="omit",
        )

        target = drug_df["target"].dropna()

        rows.append({
            "dataset": "CTRPv2",
            "classifier_source": classifier_source,
            "score": score,
            "drug_id": master_cpd_id,
            "drug": drug,
            "target": (
                target.iloc[0]
                if len(target)
                else pd.NA
            ),
            "n_models": n_models,
            "n_tissues": n_tissues,
            "beta_tissue_adjusted": (
                fit.params["score_within_tissue"]
            ),
            "standard_error_HC3": (
                fit.bse["score_within_tissue"]
            ),
            "pvalue": (
                fit.pvalues["score_within_tissue"]
            ),
            "spearman_rho_within_tissue": rho,
            "spearman_pvalue_within_tissue": rho_p,
            "median_auc": data["auc"].median(),
        })

    results = pd.DataFrame(rows)

    if results.empty:
        return results

    results["FDR_within_classifier"] = np.nan

    for _, idx in results.groupby(
        "classifier_source"
    ).groups.items():
        idx = list(idx)

        results.loc[
            idx,
            "FDR_within_classifier",
        ] = multipletests(
            results.loc[idx, "pvalue"],
            method="fdr_bh",
        )[1]

    results["FDR_global_CTRP"] = multipletests(
        results["pvalue"],
        method="fdr_bh",
    )[1]

    results["direction"] = np.where(
        results["beta_tissue_adjusted"] < 0,
        "sensitivity",
        "resistance",
    )

    return results.sort_values(
        [
            "FDR_within_classifier",
            "pvalue",
        ]
    ).reset_index(drop=True)


ctrp_results = run_ctrp_continuous_screen(
    ctrp_scores
)

display(
    ctrp_results.groupby("classifier_source").agg(
        n_tests=("drug_id", "size"),
        n_FDR05=(
            "FDR_within_classifier",
            lambda x: (x < 0.05).sum(),
        ),
        min_FDR=("FDR_within_classifier", "min"),
    )
)

display(ctrp_results.head(60))

ctrp_results.to_csv(
    OUTPUT_DIR / "ctrp_continuous_tissue_adjusted_results.csv",
    index=False,
)


In [ ]:

significant_ctrp = ctrp_results[
    ctrp_results["FDR_within_classifier"] < 0.05
].copy()

display(
    significant_ctrp[
        [
            "classifier_source",
            "drug",
            "target",
            "n_models",
            "n_tissues",
            "beta_tissue_adjusted",
            "standard_error_HC3",
            "pvalue",
            "FDR_within_classifier",
            "FDR_global_CTRP",
            "direction",
        ]
    ].head(100)
)

significant_ctrp.to_csv(
    OUTPUT_DIR / "ctrp_significant_results.csv",
    index=False,
)


In [ ]:

MEK_CANDIDATES = [
    "trametinib",
    "selumetinib",
    "refametinib",
    "PD0325901",
]

mek_keys = {
    normalize_drug_name(x): x
    for x in MEK_CANDIDATES
}

mek_ctrp = ctrp_results.copy()
mek_ctrp["drug_key"] = (
    mek_ctrp["drug"].map(normalize_drug_name)
)

mek_ctrp = mek_ctrp[
    mek_ctrp["drug_key"].isin(mek_keys)
].copy()

mek_ctrp["requested_name"] = (
    mek_ctrp["drug_key"].map(mek_keys)
)

display(
    mek_ctrp[
        [
            "requested_name",
            "drug",
            "classifier_source",
            "n_models",
            "n_tissues",
            "beta_tissue_adjusted",
            "pvalue",
            "FDR_within_classifier",
            "direction",
        ]
    ].sort_values(
        [
            "requested_name",
            "classifier_source",
        ]
    )
)

available_keys = set(
    ctrp["drug"].map(normalize_drug_name)
)

mek_availability = pd.DataFrame({
    "drug": MEK_CANDIDATES,
})

mek_availability["available_in_CTRP"] = (
    mek_availability["drug"]
    .map(normalize_drug_name)
    .isin(available_keys)
)

display(mek_availability)

mek_ctrp.to_csv(
    OUTPUT_DIR / "mek_inhibitor_ctrp_validation.csv",
    index=False,
)


In [ ]:

PREVIOUS_CANDIDATES = [
    "beta-lapachone",
    "tasisulam",
    "sangivamycin",
    "lacitol",
    "LY2801653",
    "pilaralisib",
    "indisulam",
    "M-344",
    "oligomycin A",
    "dofetilide",
    "FG-4592",
    "SB-228357",
    "taselisib",
    "buparlisib",
    "trametinib",
    "selumetinib",
    "refametinib",
    "PD0325901",
    "afatinib",
    "gefitinib",
    "bosutinib",
    "XAV939",
    "palbociclib",
    "ribociclib",
    "abemaciclib",
    "MK-1775",
    "LY2603618",
    "sulfasalazine",
    "erastin",
]

candidate_keys = {
    normalize_drug_name(x): x
    for x in PREVIOUS_CANDIDATES
}

candidate_results = ctrp_results.copy()
candidate_results["drug_key"] = (
    candidate_results["drug"].map(
        normalize_drug_name
    )
)

candidate_results = candidate_results[
    candidate_results["drug_key"].isin(candidate_keys)
].copy()

candidate_results["requested_name"] = (
    candidate_results["drug_key"].map(candidate_keys)
)

display(
    candidate_results[
        [
            "requested_name",
            "drug",
            "classifier_source",
            "target",
            "n_models",
            "n_tissues",
            "beta_tissue_adjusted",
            "pvalue",
            "FDR_within_classifier",
            "direction",
        ]
    ].sort_values(
        [
            "requested_name",
            "classifier_source",
        ]
    )
)

availability = pd.DataFrame({
    "candidate": PREVIOUS_CANDIDATES,
})

availability["available_in_CTRP"] = (
    availability["candidate"]
    .map(normalize_drug_name)
    .isin(available_keys)
)

display(availability)

candidate_results.to_csv(
    OUTPUT_DIR / "previous_candidate_ctrp_validation.csv",
    index=False,
)

availability.to_csv(
    OUTPUT_DIR / "previous_candidate_ctrp_availability.csv",
    index=False,
)


In [ ]:
print(gdsc_results.columns.tolist())

In [ ]:
GDSC_FILE = first_existing(
    GDSC_RESULT_CANDIDATES
)

print("GDSC result file:", GDSC_FILE)

if GDSC_FILE is not None:
    gdsc_results = pd.read_csv(GDSC_FILE)

    
    
    gdsc_results = gdsc_results[
        (gdsc_results["analysis_set"] == "all_lineages")
        & (gdsc_results["score"] == SCORE_COLUMN)
    ].copy()

    gdsc_results["drug_key"] = (
        gdsc_results["drug"].map(
            normalize_drug_name
        )
    )

    
    
    gdsc_results = gdsc_results.rename(
        columns={
            "FDR_within_analysis":
                "FDR_within_dataset_classifier"
        }
    )

    print(
        gdsc_results.groupby(
            ["dataset", "classifier_source"]
        ).size()
    )

else:
    gdsc_results = pd.DataFrame()


In [ ]:

PRISM_FILE = first_existing(
    PRISM_RESULT_CANDIDATES
)

print("PRISM result file:", PRISM_FILE)

if PRISM_FILE is not None:
    prism_results = pd.read_csv(PRISM_FILE)

    if "analysis_set" in prism_results.columns:
        prism_results = prism_results[
            prism_results["analysis_set"]
            == "all_lineages"
        ].copy()

    if "score" in prism_results.columns:
        prism_results = prism_results[
            prism_results["score"]
            == SCORE_COLUMN
        ].copy()

    if (
        "drug" not in prism_results.columns
        and "drug_name" in prism_results.columns
    ):
        prism_results = prism_results.rename(
            columns={"drug_name": "drug"}
        )

    prism_results["drug_key"] = (
        prism_results["drug"].map(
            normalize_drug_name
        )
    )

    prism_fdr_col = next(
        (
            c for c in [
                "FDR_within_analysis",
                "FDR_global",
                "FDR_global_all_tests",
            ]
            if c in prism_results.columns
        ),
        None,
    )

    keep = [
        "classifier_source",
        "drug_key",
        "drug",
        "beta_tissue_adjusted",
        "pvalue",
    ]

    if prism_fdr_col is not None:
        keep.append(prism_fdr_col)

    prism_small = (
        prism_results[keep]
        .drop_duplicates(
            [
                "classifier_source",
                "drug_key",
            ]
        )
    )

    ctrp_small2 = ctrp_results.copy()
    ctrp_small2["drug_key"] = (
        ctrp_small2["drug"].map(
            normalize_drug_name
        )
    )

    prism_ctrp = prism_small.merge(
        ctrp_small2,
        on=[
            "classifier_source",
            "drug_key",
        ],
        how="inner",
        suffixes=("_PRISM", "_CTRP"),
    )

    prism_ctrp["same_direction"] = (
        np.sign(
            prism_ctrp["beta_tissue_adjusted_PRISM"]
        )
        ==
        np.sign(
            prism_ctrp["beta_tissue_adjusted_CTRP"]
        )
    )

    prism_ctrp["both_sensitivity"] = (
        (prism_ctrp["beta_tissue_adjusted_PRISM"] < 0)
        &
        (prism_ctrp["beta_tissue_adjusted_CTRP"] < 0)
    )

    display(
        prism_ctrp.groupby(
            "classifier_source"
        ).agg(
            n_common_drugs=("drug_key", "size"),
            same_direction=("same_direction", "sum"),
            both_sensitivity=("both_sensitivity", "sum"),
        )
    )

    prism_ctrp.to_csv(
        OUTPUT_DIR / "prism_ctrp_consistency.csv",
        index=False,
    )

else:
    prism_results = pd.DataFrame()
    prism_ctrp = pd.DataFrame()
    prism_fdr_col = None
    print("No PRISM result file found.")


In [ ]:

platform_tables = []


x = ctrp_results.copy()
x["drug_key"] = x["drug"].map(normalize_drug_name)

x = x[
    [
        "classifier_source",
        "drug_key",
        "drug",
        "beta_tissue_adjusted",
        "pvalue",
        "FDR_within_classifier",
    ]
].rename(
    columns={
        "drug": "drug_CTRP",
        "beta_tissue_adjusted": "beta_CTRP",
        "pvalue": "pvalue_CTRP",
        "FDR_within_classifier": "FDR_CTRP",
    }
)

platform_tables.append(x)


if not gdsc_results.empty:
    for dataset in ["GDSC1", "GDSC2"]:
        g = gdsc_results[
            gdsc_results["dataset"] == dataset
        ].copy()

        if g.empty:
            continue

        g["drug_key"] = (
            g["drug"].map(normalize_drug_name)
        )

        g = g[
            [
                "classifier_source",
                "drug_key",
                "drug",
                "beta_tissue_adjusted",
                "pvalue",
                "FDR_within_dataset_classifier",
            ]
        ].rename(
            columns={
                "drug": f"drug_{dataset}",
                "beta_tissue_adjusted": f"beta_{dataset}",
                "pvalue": f"pvalue_{dataset}",
                "FDR_within_dataset_classifier": f"FDR_{dataset}",
            }
        )

        platform_tables.append(g)


if not prism_results.empty:
    p = prism_results.copy()
    p["drug_key"] = p["drug"].map(normalize_drug_name)

    keep = [
        "classifier_source",
        "drug_key",
        "drug",
        "beta_tissue_adjusted",
        "pvalue",
    ]

    if prism_fdr_col is not None:
        keep.append(prism_fdr_col)

    p = p[keep].drop_duplicates(
        [
            "classifier_source",
            "drug_key",
        ]
    )

    rename = {
        "drug": "drug_PRISM",
        "beta_tissue_adjusted": "beta_PRISM",
        "pvalue": "pvalue_PRISM",
    }

    if prism_fdr_col is not None:
        rename[prism_fdr_col] = "FDR_PRISM"

    p = p.rename(columns=rename)
    platform_tables.append(p)


consensus = None

for table in platform_tables:
    table = table.drop_duplicates(
        [
            "classifier_source",
            "drug_key",
        ]
    )

    if consensus is None:
        consensus = table.copy()
    else:
        consensus = consensus.merge(
            table,
            on=[
                "classifier_source",
                "drug_key",
            ],
            how="outer",
        )


drug_name_cols = [
    c for c in consensus.columns
    if c.startswith("drug_")
]

consensus["drug"] = (
    consensus[drug_name_cols]
    .bfill(axis=1)
    .iloc[:, 0]
)

beta_cols = [
    c for c in [
        "beta_PRISM",
        "beta_GDSC1",
        "beta_GDSC2",
        "beta_CTRP",
    ]
    if c in consensus.columns
]


def summarize_direction(row):
    values = [
        row[c]
        for c in beta_cols
        if pd.notna(row[c])
    ]

    if not values:
        return pd.Series({
            "n_platforms": 0,
            "n_sensitivity": 0,
            "n_resistance": 0,
            "all_same_direction": False,
        })

    signs = np.sign(values)

    return pd.Series({
        "n_platforms": len(values),
        "n_sensitivity": sum(v < 0 for v in values),
        "n_resistance": sum(v > 0 for v in values),
        "all_same_direction": len(set(signs)) == 1,
    })


summary = consensus.apply(
    summarize_direction,
    axis=1,
)

consensus = pd.concat(
    [consensus, summary],
    axis=1,
)

consensus = consensus.sort_values(
    [
        "n_sensitivity",
        "all_same_direction",
        "n_platforms",
    ],
    ascending=[False, False, False],
)

display(consensus.head(100))

consensus.to_csv(
    OUTPUT_DIR / "prism_gdsc_ctrp_consensus.csv",
    index=False,
)


In [ ]:

meta = consensus.copy()

platform_specs = []

for platform in [
    "PRISM",
    "GDSC1",
    "GDSC2",
    "CTRP",
]:
    beta_col = f"beta_{platform}"
    p_col = f"pvalue_{platform}"

    if (
        beta_col in meta.columns
        and p_col in meta.columns
    ):
        platform_specs.append(
            (platform, beta_col, p_col)
        )


def signed_z(beta, p):
    if pd.isna(beta) or pd.isna(p):
        return np.nan

    p = float(np.clip(p, 1e-300, 1.0))

    return (
        np.sign(beta)
        * norm.isf(p / 2.0)
    )


for platform, beta_col, p_col in platform_specs:
    meta[f"z_{platform}"] = [
        signed_z(beta, p)
        for beta, p in zip(
            meta[beta_col],
            meta[p_col],
        )
    ]


z_cols = [
    f"z_{platform}"
    for platform, _, _ in platform_specs
]


def combine_row(row):
    z_values = np.array(
        [
            row[c]
            for c in z_cols
            if pd.notna(row[c])
        ],
        dtype=float,
    )

    if len(z_values) < 2:
        return pd.Series({
            "meta_n_platforms": len(z_values),
            "meta_z": np.nan,
            "meta_pvalue": np.nan,
        })

    z = z_values.sum() / np.sqrt(len(z_values))
    p = 2 * norm.sf(abs(z))

    return pd.Series({
        "meta_n_platforms": len(z_values),
        "meta_z": z,
        "meta_pvalue": p,
    })


combined = meta.apply(
    combine_row,
    axis=1,
)

meta = pd.concat(
    [meta, combined],
    axis=1,
)

valid = meta["meta_pvalue"].notna()

meta["meta_FDR"] = np.nan

if valid.any():
    meta.loc[
        valid,
        "meta_FDR",
    ] = multipletests(
        meta.loc[valid, "meta_pvalue"],
        method="fdr_bh",
    )[1]

meta["meta_direction"] = np.where(
    meta["meta_z"] < 0,
    "sensitivity",
    "resistance",
)

meta = meta.sort_values(
    [
        "meta_FDR",
        "meta_pvalue",
    ],
    na_position="last",
)

display(
    meta[
        [
            "classifier_source",
            "drug",
            "meta_n_platforms",
            "meta_z",
            "meta_pvalue",
            "meta_FDR",
            "meta_direction",
            "n_sensitivity",
            "n_resistance",
            "all_same_direction",
        ]
    ].head(100)
)

meta.to_csv(
    OUTPUT_DIR / "cross_platform_exploratory_stouffer_meta.csv",
    index=False,
)


In [ ]:

coverage = pd.DataFrame({
    "metric": [
        "CTRP cell lines total",
        "CTRP cell lines mapped to DepMap",
        "CTRP compounds",
        "CTRP model-drug response rows",
        "CRC score models in CTRP analysis",
        "UCEC score models in CTRP analysis",
    ],
    "value": [
        ctrp["master_ccl_id"].nunique(),
        ctrp_mapped["DepMap_ID"].nunique(),
        ctrp["master_cpd_id"].nunique(),
        len(ctrp),
        ctrp_scores.loc[
            ctrp_scores["classifier_source"] == "CRC",
            "DepMap_ID",
        ].nunique(),
        ctrp_scores.loc[
            ctrp_scores["classifier_source"] == "UCEC",
            "DepMap_ID",
        ].nunique(),
    ],
})

display(coverage)

coverage.to_csv(
    OUTPUT_DIR / "ctrp_coverage_summary.csv",
    index=False,
)

print("\nOutput files:")
for path in sorted(OUTPUT_DIR.glob("*.csv")):
    print(" -", path.name)
